# Eksplorasi Dataset - Plate Count Reader

Notebook ini melakukan eksplorasi mendalam terhadap dataset AGAR yang digunakan dalam proyek
**Automated Plate Count Reader** untuk AI Open Innovation Challenge 2026.

## Tujuan
1. Memahami struktur dan isi dataset AGAR
2. Memvisualisasikan sampel gambar dari setiap subset
3. Menganalisis distribusi jumlah koloni
4. Memeriksa kualitas mask segmentasi U2Net
5. Mengumpulkan statistik dataset untuk persiapan training

## Parameter Utama (from train_18k_gpu1.py)
- **Output**: `data/yolo_18k_multiclass/`
- **Kelas**: colony (0), bubble (1), dust (2), crack (3)
- **Split**: 80/12/8 (train/val/test)
- **PL_CONF**: 0.20
- **Synthetic**: 1000 bubble, 800 dust, 500 crack

## Setup Environment

In [ ]:
# Install dependencies jika belum ada
import subprocess
import sys

deps = ['matplotlib', 'numpy', 'opencv-python', 'Pillow', 'scipy']
for dep in deps:
    try:
        __import__(dep.replace('-', '_').split('[')[0])
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', dep, '-q'])
print('Semua dependensi terinstall')

In [ ]:
# Import libraries
import os
import glob
import json
import random
from pathlib import Path
from collections import Counter, defaultdict

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from scipy import ndimage

# Settings
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12
random.seed(42)
np.random.seed(42)

print('Libraries berhasil diimport')

## Struktur Dataset AGAR

Dataset AGAR terdiri dari dua subset utama:
- **dataset_for_u2net**: 255 gambar dengan mask segmentasi (untuk deteksi koloni)
- **dataset_for_resnet**: ~18K gambar klasifikasi berdasarkan jumlah koloni (0-9)

**PENTING**: Struktur direktori memiliki nesting ganda:
- `dataset_for_u2net/dataset_for_u2net/{train,valid,train_mask,valid_mask}/`
- `dataset_for_resnet/dataset_for_resnet/{train,val}/`

In [ ]:
# Konfigurasi path dataset (SESUAI train_18k_gpu1.py)
WORKSPACE = Path('/media/lambda_one/DFSSD04/project/healtcare')
DATA_DIR = WORKSPACE / 'data' / 'agar' / 'dataset'

# Path dengan nested structure (dataset_for_u2net/dataset_for_u2net/)
U2NET_DIR = DATA_DIR / 'dataset_for_u2net' / 'dataset_for_u2net'
RESNET_DIR = DATA_DIR / 'dataset_for_resnet' / 'dataset_for_resnet'

# Output dataset YOLO (SESUAI train_18k_gpu1.py)
YOLO_OUTPUT = WORKSPACE / 'data' / 'yolo_18k_multiclass'

# Verifikasi path exists
print(f'Workspace: {WORKSPACE}')
print(f'Data dir exists: {DATA_DIR.exists()}')
print(f'U2Net dir exists: {U2NET_DIR.exists()}')
print(f'ResNet dir exists: {RESNET_DIR.exists()}')
print(f'YOLO output exists: {YOLO_OUTPUT.exists()}')

In [ ]:
# Tampilkan struktur direktori
def show_tree(directory, max_depth=3, prefix=''):
    """Tampilkan struktur direktori seperti tree command."""
    if not directory.exists():
        print(f'Directory tidak ditemukan: {directory}')
        return
    
    try:
        items = sorted(directory.iterdir())
    except PermissionError:
        print(f'{prefix}[permission denied]')
        return
    
    dirs = [x for x in items if x.is_dir()]
    files = [x for x in items if x.is_file()]
    
    # Show dirs first (limit to 10 for brevity)
    for i, d in enumerate(dirs[:10]):
        is_last = (i == len(dirs[:10]) - 1) and len(files) == 0
        connector = '--- ' if is_last else '|-- '
        print(f'{prefix}{connector}{d.name}/')
        if max_depth > 1:
            extension = '    ' if is_last else '|   '
            show_tree(d, max_depth - 1, prefix + extension)
    
    if len(dirs) > 10:
        print(f'{prefix}|-- ... ({len(dirs) - 10} direktori lainnya)')
    
    # Show files (limit to 5)
    for i, f in enumerate(files[:5]):
        is_last = i == len(files[:5]) - 1
        connector = '--- ' if is_last else '|-- '
        print(f'{prefix}{connector}{f.name}')
    
    if len(files) > 5:
        print(f'{prefix}--- ... ({len(files) - 5} file lainnya)')

print('Struktur Dataset U2Net:')
print('dataset_for_u2net/dataset_for_u2net/')
show_tree(U2NET_DIR, max_depth=2, prefix='')
print()
print('Struktur Dataset ResNet:')
print('dataset_for_resnet/dataset_for_resnet/')
show_tree(RESNET_DIR, max_depth=2, prefix='')

In [ ]:
# Hitung file di setiap subset
def count_images(directory, extensions=('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')):
    """Hitung jumlah file gambar dalam direktori (recursive)."""
    count = 0
    for ext in extensions:
        count += len(list(directory.rglob(f'*{ext}')))
    return count

u2net_img_count = count_images(U2NET_DIR)
resnet_img_count = count_images(RESNET_DIR)

print(f'Jumlah Gambar per Subset:')
print(f'  dataset_for_u2net : {u2net_img_count} gambar')
print(f'  dataset_for_resnet: {resnet_img_count} gambar')
print(f'  Total             : {u2net_img_count + resnet_img_count} gambar')

## Visualisasi Sample Gambar

In [ ]:
# Fungsi helper untuk menampilkan gambar
def display_images(images, titles=None, cols=4, figsize=(16, 4), cmap=None):
    """Tampilkan multiple images dalam grid."""
    rows = (len(images) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    if rows == 1:
        axes = [axes] if cols == 1 else axes
    else:
        axes = axes.flatten()
    
    for idx, ax in enumerate(axes):
        if idx < len(images):
            if isinstance(images[idx], str):
                img = cv2.imread(images[idx])
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            else:
                img = images[idx]
            ax.imshow(img, cmap=cmap)
            if titles and idx < len(titles):
                ax.set_title(titles[idx], fontsize=10)
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

# Sample gambar dari U2Net dataset
u2net_train_dir = U2NET_DIR / 'train'
u2net_valid_dir = U2NET_DIR / 'valid'

u2net_train_images = sorted(list(u2net_train_dir.glob('*.jpg'))) + sorted(list(u2net_train_dir.glob('*.png')))
u2net_valid_images = sorted(list(u2net_valid_dir.glob('*.jpg'))) + sorted(list(u2net_valid_dir.glob('*.png')))

print(f'U2Net train images: {len(u2net_train_images)}')
print(f'U2Net valid images: {len(u2net_valid_images)}')

# Tampilkan 4 sample
sample_images = random.sample(u2net_train_images, min(4, len(u2net_train_images)))
display_images(
    [str(p) for p in sample_images],
    titles=[p.name for p in sample_images],
    cols=4,
    figsize=(16, 4)
)

In [ ]:
# Sample gambar dari ResNet dataset (per jumlah koloni)
resnet_train_dir = RESNET_DIR / 'train'
count_dirs = sorted([d for d in resnet_train_dir.iterdir() if d.is_dir()], key=lambda x: int(x.name))

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for idx, count_dir in enumerate(count_dirs):
    row, col = idx // 5, idx % 5
    images = list(count_dir.glob('*.jpg')) + list(count_dir.glob('*.png'))
    if images:
        sample = random.choice(images)
        img = cv2.imread(str(sample))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[row, col].imshow(img)
        axes[row, col].set_title(f'{count_dir.name} koloni\n({len(images)} gambar)', fontsize=11)
    axes[row, col].axis('off')

plt.suptitle('Sample Gambar Dataset ResNet per Jumlah Koloni', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Distribusi Jumlah Koloni

In [ ]:
# Distribusi jumlah koloni di ResNet dataset
# SESUAI train_18k_gpu1.py: ResNet punya split 'train' dan 'val'
train_counts = {}
val_counts = {}

for split_name, count_dict in [('train', train_counts), ('val', val_counts)]:
    split_dir = RESNET_DIR / split_name
    if split_dir.exists():
        for count_dir in sorted(split_dir.iterdir(), key=lambda x: int(x.name) if x.is_dir() else -1):
            if count_dir.is_dir():
                imgs = list(count_dir.glob('*.jpg')) + list(count_dir.glob('*.png'))
                count_dict[int(count_dir.name)] = len(imgs)

# Bar chart
fig, ax = plt.subplots(figsize=(12, 6))
x = list(range(10))
train_vals = [train_counts.get(i, 0) for i in x]
val_vals = [val_counts.get(i, 0) for i in x]

width = 0.35
bars1 = ax.bar([i - width/2 for i in x], train_vals, width, label='Train', color='#2196F3', alpha=0.8)
bars2 = ax.bar([i + width/2 for i in x], val_vals, width, label='Validation', color='#FF9800', alpha=0.8)

ax.set_xlabel('Jumlah Koloni', fontsize=12)
ax.set_ylabel('Jumlah Gambar', fontsize=12)
ax.set_title('Distribusi Jumlah Koloni pada Dataset ResNet', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([str(i) for i in x])
ax.legend()

for bar in bars1:
    height = bar.get_height()
    if height > 0:
        ax.text(bar.get_x() + bar.get_width()/2., height, f'{int(height)}',
                ha='center', va='bottom', fontsize=8)
for bar in bars2:
    height = bar.get_height()
    if height > 0:
        ax.text(bar.get_x() + bar.get_width()/2., height, f'{int(height)}',
                ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

print(f'Total Train: {sum(train_vals)} gambar')
print(f'Total Val  : {sum(val_vals)} gambar')

## Analisis Mask U2Net

Dataset U2Net menyediakan mask segmentasi biner (255 = koloni, 0 = background).
Struktur mask: `dataset_for_u2net/dataset_for_u2net/{train,valid}_mask/colony_detecting/`

In [ ]:
# Cari mask files di U2Net dataset (SESUAI train_18k_gpu1.py)
# Struktur: U2NET_DIR / '{split}_mask' / 'colony_detecting' / '*_mask.png'
mask_dirs = []
for split_name in ['train', 'valid']:
    mask_dir = U2NET_DIR / f'{split_name}_mask' / 'colony_detecting'
    if mask_dir.exists():
        mask_dirs.append(mask_dir)
        mask_files = list(mask_dir.glob('*_mask.png'))
        print(f'{split_name}_mask/colony_detecting: {len(mask_files)} mask files')

# Juga cari secara umum sebagai fallback
if not mask_dirs:
    for root, dirs, files in os.walk(U2NET_DIR):
        for d in dirs:
            if 'mask' in d.lower():
                md = Path(root) / d
                mf = list(md.glob('*.jpg')) + list(md.glob('*.png')) + list(md.glob('*.bmp'))
                if mf:
                    mask_dirs.append(md)
                    print(f'  {md.relative_to(U2NET_DIR)}: {len(mf)} mask files')

print(f'\nTotal direktori mask ditemukan: {len(mask_dirs)}')

In [ ]:
# Visualisasi overlay mask pada gambar asli
def overlay_mask(image_path, mask_path, alpha=0.4):
    """Overlay mask segmentasi pada gambar asli."""
    img = cv2.imread(str(image_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    
    # Resize mask jika ukuran berbeda
    if mask.shape[:2] != img.shape[:2]:
        mask = cv2.resize(mask, (img.shape[1], img.shape[0]))
    
    # Buat overlay merah pada area koloni
    overlay = img.copy()
    overlay[mask > 127] = [255, 0, 0]  # Red for colonies
    
    # Blend
    result = cv2.addWeighted(img, 1 - alpha, overlay, alpha, 0)
    return img, mask, result

# Cari pasangan gambar-mask
if mask_dirs:
    primary_mask_dir = mask_dirs[0]
    mask_files = sorted(list(primary_mask_dir.glob('*.jpg')) + list(primary_mask_dir.glob('*.png')) + list(primary_mask_dir.glob('*.bmp')))
    
    sample_masks = random.sample(mask_files, min(3, len(mask_files)))
    
    fig, axes = plt.subplots(len(sample_masks), 3, figsize=(15, 5 * len(sample_masks)))
    if len(sample_masks) == 1:
        axes = axes.reshape(1, -1)
    
    for idx, mask_path in enumerate(sample_masks):
        # Cari gambar asli - sesuai train_18k_gpu1.py logic
        img_name = mask_path.name.replace('_mask.png', '.png')
        img_path = None
        for split in ['train', 'valid']:
            candidate = U2NET_DIR / split / img_name
            if candidate.exists():
                img_path = candidate
                break
        
        if img_path is None:
            img_name_stem = mask_path.stem.replace('_mask', '')
            for split in ['train', 'valid']:
                for ext in ['.jpg', '.png', '.bmp']:
                    candidate = U2NET_DIR / split / (img_name_stem + ext)
                    if candidate.exists():
                        img_path = candidate
                        break
                if img_path:
                    break
        
        try:
            img, mask, overlay = overlay_mask(img_path, mask_path)
            axes[idx, 0].imshow(img)
            axes[idx, 0].set_title('Gambar Asli', fontsize=11)
            axes[idx, 1].imshow(mask, cmap='gray')
            axes[idx, 1].set_title('Mask Segmentasi', fontsize=11)
            axes[idx, 2].imshow(overlay)
            axes[idx, 2].set_title('Overlay', fontsize=11)
        except Exception as e:
            axes[idx, 0].set_title(f'Error: {e}')
        
        for ax in axes[idx]:
            ax.axis('off')
    
    plt.suptitle('Analisis Mask Segmentasi U2Net', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('Tidak ada direktori mask ditemukan. Mencari file mask secara langsung...')
    all_mask_files = list(U2NET_DIR.rglob('*mask*'))
    print(f"File dengan 'mask' di nama: {len(all_mask_files)}")

In [ ]:
# Analisis ukuran connected components dalam mask
if mask_dirs and len(mask_files) > 0:
    MIN_COLONY_AREA = 8  # SESUAI train_18k_gpu1.py
    num_colonies_per_mask = []
    avg_colony_sizes = []
    
    sample_for_analysis = random.sample(mask_files, min(20, len(mask_files)))
    
    for mask_path in sample_for_analysis:
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            continue
        
        binary = (mask > 127).astype(np.uint8)
        labeled, num_features = ndimage.label(binary)
        num_colonies_per_mask.append(num_features)
        
        if num_features > 0:
            for lid in range(1, num_features + 1):
                comp = labeled == lid
                if comp.sum() >= MIN_COLONY_AREA:
                    ys, xs = np.where(comp)
                    avg_colony_sizes.append(len(xs))
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    if num_colonies_per_mask:
        ax1.hist(num_colonies_per_mask, bins=range(0, max(num_colonies_per_mask + [10]) + 2), 
                 color='#2196F3', alpha=0.7, edgecolor='black')
        ax1.set_xlabel('Jumlah Koloni (Connected Components)')
        ax1.set_ylabel('Frekuensi')
        ax1.set_title('Distribusi Jumlah Koloni per Mask', fontweight='bold')
        ax1.axvline(np.mean(num_colonies_per_mask), color='red', linestyle='--', 
                    label=f'Rata-rata: {np.mean(num_colonies_per_mask):.1f}')
        ax1.legend()
    
    if avg_colony_sizes:
        ax2.hist(avg_colony_sizes, bins=30, color='#4CAF50', alpha=0.7, edgecolor='black')
        ax2.set_xlabel('Ukuran Koloni (pixels)')
        ax2.set_ylabel('Frekuensi')
        ax2.set_title('Distribusi Ukuran Koloni dalam Mask', fontweight='bold')
        ax2.axvline(np.mean(avg_colony_sizes), color='red', linestyle='--',
                    label=f'Rata-rata: {np.mean(avg_colony_sizes):.0f} px')
        ax2.axvline(MIN_COLONY_AREA, color='orange', linestyle=':',
                    label=f'MIN_COLONY_AREA: {MIN_COLONY_AREA} px')
        ax2.legend()
    
    plt.tight_layout()
    plt.show()
    
    print(f'Rata-rata koloni per mask: {np.mean(num_colonies_per_mask):.1f}')
    print(f'Min/Max koloni per mask: {min(num_colonies_per_mask)}/{max(num_colonies_per_mask)}')
    if avg_colony_sizes:
        print(f'Rata-rata ukuran koloni: {np.mean(avg_colony_sizes):.0f} pixels')
else:
    print('Tidak ada mask file tersedia untuk analisis')

## Statistik Dataset YOLO (jika sudah disiapkan)

In [ ]:
# Cek dataset YOLO yang sudah disiapkan oleh train_18k_gpu1.py
if YOLO_OUTPUT.exists():
    print('Dataset YOLO 18k_multiclass sudah ada:')
    for split in ['train', 'val', 'test']:
        img_dir = YOLO_OUTPUT / split / 'images'
        lbl_dir = YOLO_OUTPUT / split / 'labels'
        if img_dir.exists():
            n_img = len(list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png')))
            n_lbl = len(list(lbl_dir.glob('*.txt'))) if lbl_dir.exists() else 0
            print(f'  {split}: {n_img} images, {n_lbl} labels')
    
    import yaml
    yaml_path = YOLO_OUTPUT / 'data.yaml'
    if yaml_path.exists():
        with open(str(yaml_path)) as f:
            data_cfg = yaml.safe_load(f)
        print(f'\n  data.yaml: nc={data_cfg.get("nc")}, names={data_cfg.get("names")}')
        print(f'  path={data_cfg.get("path")}')
else:
    print('Dataset YOLO belum disiapkan. Jalankan 02_data_preparation.ipynb atau train_18k_gpu1.py')

## Statistik Dimensi Gambar

In [ ]:
# Kumpulkan statistik dimensi gambar
def get_image_stats(directory, max_samples=200):
    """Kumpulkan statistik dimensi gambar."""
    images = list(directory.rglob('*.jpg')) + list(directory.rglob('*.png'))
    sample = random.sample(images, min(max_samples, len(images)))
    
    widths, heights = [], []
    for img_path in sample:
        try:
            img = Image.open(str(img_path))
            w, h = img.size
            widths.append(w)
            heights.append(h)
        except Exception:
            pass
    
    return widths, heights

print('Mengumpulkan statistik dimensi gambar...')
u2net_widths, u2net_heights = get_image_stats(U2NET_DIR)
resnet_widths, resnet_heights = get_image_stats(RESNET_DIR)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if u2net_widths:
    axes[0].scatter(u2net_widths, u2net_heights, alpha=0.5, s=20, color='#2196F3')
    axes[0].set_xlabel('Width (pixels)')
    axes[0].set_ylabel('Height (pixels)')
    axes[0].set_title(f'U2Net Dataset - Dimensi Gambar\n(n={len(u2net_widths)})', fontweight='bold')

if resnet_widths:
    axes[1].scatter(resnet_widths, resnet_heights, alpha=0.5, s=20, color='#4CAF50')
    axes[1].set_xlabel('Width (pixels)')
    axes[1].set_ylabel('Height (pixels)')
    axes[1].set_title(f'ResNet Dataset - Dimensi Gambar\n(n={len(resnet_widths)})', fontweight='bold')

plt.tight_layout()
plt.show()

print('\nStatistik Dimensi Gambar:')
print(f'{"Subset":<20} {"Width (min-max)":<25} {"Height (min-max)":<25} {"Rata-rata"}')
print('-' * 90)
if u2net_widths:
    print(f'{"U2Net":<20} {min(u2net_widths)}-{max(u2net_widths):<20} {min(u2net_heights)}-{max(u2net_heights):<20} {np.mean(u2net_widths):.0f}x{np.mean(u2net_heights):.0f}')
if resnet_widths:
    print(f'{"ResNet":<20} {min(resnet_widths)}-{max(resnet_widths):<20} {min(resnet_heights)}-{max(resnet_heights):<20} {np.mean(resnet_widths):.0f}x{np.mean(resnet_heights):.0f}')

## Ringkasan

In [ ]:
# Tabel ringkasan dataset
print('=' * 70)
print('RINGKASAN EKSPLORASI DATASET AGAR')
print('=' * 70)
print()
print(f'{"Parameter":<35} {"Nilai"}')
print('-' * 70)
print(f'{"Total gambar U2Net":<35} {u2net_img_count}')
print(f'{"Total gambar ResNet":<35} {resnet_img_count}')
print(f'{"Total keseluruhan":<35} {u2net_img_count + resnet_img_count}')
print(f'{"Kelas koloni (ResNet)":<35} 0 - 9 (10 kelas jumlah)')
print(f'{"Mask segmentasi (U2Net)":<35} {"Tersedia" if mask_dirs else "Tidak tersedia"}')
if u2net_widths:
    print(f'{"Dimensi U2Net (rata-rata)":<35} {np.mean(u2net_widths):.0f} x {np.mean(u2net_heights):.0f}')
if resnet_widths:
    print(f'{"Dimensi ResNet (rata-rata)":<35} {np.mean(resnet_widths):.0f} x {np.mean(resnet_heights):.0f}')
print(f'{"Total train (ResNet)":<35} {sum(train_vals)}')
print(f'{"Total val (ResNet)":<35} {sum(val_vals)}')
print()
print('Catatan Penting:')
print('  - Dataset U2Net memiliki mask biner yang bisa dikonversi ke YOLO bbox')
print('  - Dataset ResNet perlu pseudo-labeling (CONF=0.20, sesuai train_18k_gpu1.py)')
print('  - Perlu generasi artefak sintetis: 1000 bubble, 800 dust, 500 crack')
print('  - Output dataset: data/yolo_18k_multiclass/')
print('  - Split ratio: 80/12/8 (train/val/test)')
print('=' * 70)